# epileval — Forecasting (alarm-based) Quick Start

Alarm-based metrics for seizure forecasting with an **explicit** `AlarmPolicy`. Every reproducibility knob (SPH / SOP / cadence / refractory / alarm threshold / FP-denominator) is mandatory — no silent defaults. The same probability stream that produces sample-based AUROC also produces alarm-based sensitivity / FP-per-hour / IoC, so the two regimes can be reported side-by-side.

**What this notebook covers**

1. Synthesise a 24-hour probability stream with three seizures and a pre-ictal ramp before each.
2. Pin a paper-equivalent `AlarmPolicy`.
3. Score with `forecasting.evaluate_stream` + a Poisson surrogate IoC test.
4. Threshold sweep and cadence ablation.

Companion notebook: `01_detection_quick_start.ipynb` covers the sample-based regime.

In [1]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
from epileval import AlarmPolicy, forecasting
import epileval

epileval.__version__

'0.1.0'

## 1. Synthetic 24-hour stream

Three seizures, 60-second cadence, a pre-ictal ramp from 60 min down to 10 min before each onset that lifts the predictor probability from ~0.1 noise floor up to ~0.7.

In [2]:
rng = np.random.default_rng(0)
duration = 24 * 3600
cadence = 60.0
times = np.arange(0, duration, cadence)
seizures = np.array([4 * 3600, 11 * 3600, 19 * 3600], dtype=float)

proba = rng.uniform(0, 0.3, size=times.size)
for sz in seizures:
    ramp_mask = (times >= sz - 3600) & (times < sz - 600)
    proba[ramp_mask] += np.linspace(0.1, 0.7, ramp_mask.sum())
proba = np.clip(proba, 0, 1)

print(f'samples           : {times.size}')
print(f'cadence (s)       : {cadence:.0f}')
print(f'seizures          : {[f"{s/3600:.1f} h" for s in seizures]}')
print(f'mean proba        : {proba.mean():.3f}')
print(f'pre-ictal mean    : {proba[(times >= 4*3600 - 3600) & (times < 4*3600 - 600)].mean():.3f}')

samples           : 1440
cadence (s)       : 60
seizures          : ['4.0 h', '11.0 h', '19.0 h']
mean proba        : 0.194
pre-ictal mean    : 0.547


## 2. AlarmPolicy — every knob explicit

Mormann's tradition uses `fp_denominator='interictal'` (FP/hr is normalised by interictal-only time). Set `'total'` to match Cook 2013 / Karoly 2017.

In [3]:
policy = AlarmPolicy(
    sph_seconds=300,            # seizure-prediction horizon
    sop_seconds=600,            # seizure-occurrence period
    cadence_seconds=60,         # alarm-evaluation cadence
    refractory_seconds=600,     # post-alarm refractory window
    alarm_threshold=0.6,
    fp_denominator='interictal',
)
policy

AlarmPolicy(sph_seconds=300, sop_seconds=600, cadence_seconds=60, refractory_seconds=600, alarm_threshold=0.6, merge_consecutive=True, fp_denominator='interictal')

## 3. Evaluate + IoC surrogate

`n_surrogate=500` runs the chance-baseline IoC test — same alarm count, randomly placed (Poisson by default). The reported `ioc_pvalue` is the proportion of surrogate IoC scores ≥ the model's IoC.

In [4]:
rep = forecasting.evaluate_stream(
    proba, times, seizures, policy,
    total_recording_time=float(duration),
    n_surrogate=500,
)
print(f'sensitivity            : {rep.sensitivity:.3f}')
print(f'FP per hour            : {rep.fp_per_hour:.3f}')
print(f'IoC                    : {rep.ioc:.3f}')
print(f'time-in-warning        : {rep.time_in_warning_frac:.3f}')
print(f'beats chance (alarm)   : {getattr(rep, "beats_chance_alarm", "n/a")}')

sensitivity            : 0.000
FP per hour            : 0.264
IoC                    : -0.041
time-in-warning        : 0.042
beats chance (alarm)   : n/a


## 4. Threshold sweep

`sweep_thresholds` returns a DataFrame of (threshold, sensitivity, fp_per_hour, ioc, ...) — the operating curve.

In [5]:
df_thr = forecasting.sweep_thresholds(
    proba, times, seizures, policy,
    thresholds=np.linspace(0.1, 0.9, 9),
    total_recording_time=float(duration),
)
df_thr.round(3).head()

,name,regime,roc_auc,pr_auc,brier,balanced_accuracy,mcc,sensitivity,precision,f1,...,n_ref_events,n_tp,n_fp,sph_seconds,sop_seconds,time_in_warning_frac,ioc,surrogate_sensitivity,extras,threshold
0,@thresh=0.10,forecasting,None,None,None,None,None,0.667,None,None,...,3,2,101,300,600,0.715,0.160,0.507,"{'policy': {'sph_s': 300, 'sop_s': 600, 'caden...",0.1
1,@thresh=0.20,forecasting,None,None,None,None,None,1.000,None,None,...,3,3,105,300,600,0.750,0.490,0.510,"{'policy': {'sph_s': 300, 'sop_s': 600, 'caden...",0.2
2,@thresh=0.30,forecasting,None,None,None,None,None,0.000,None,None,...,3,0,6,300,600,0.042,-0.040,0.040,"{'policy': {'sph_s': 300, 'sop_s': 600, 'caden...",0.3
3,@thresh=0.40,forecasting,None,None,None,None,None,0.000,None,None,...,3,0,5,300,600,0.035,-0.032,0.032,"{'policy': {'sph_s': 300, 'sop_s': 600, 'caden...",0.4
4,@thresh=0.50,forecasting,None,None,None,None,None,0.000,None,None,...,3,0,6,300,600,0.042,-0.040,0.040,"{'policy': {'sph_s': 300, 'sop_s': 600, 'caden...",0.5


## 5. Cadence ablation

Same predictor, varying alarm-evaluation cadence. Lower cadence = more potential FPs per hour at the same threshold.

In [6]:
policies = [
    AlarmPolicy(
        sph_seconds=300, sop_seconds=600,
        cadence_seconds=c, refractory_seconds=600,
        alarm_threshold=0.6, fp_denominator='interictal',
    )
    for c in [30, 60, 120, 300]
]
df_cad = forecasting.sweep_policies(
    proba, times, seizures, policies,
    total_recording_time=float(duration),
)
df_cad.round(3).head()

,name,regime,roc_auc,pr_auc,brier,balanced_accuracy,mcc,sensitivity,precision,f1,...,ioc,surrogate_sensitivity,extras,sph_s,sop_s,cadence_s,refractory_s,alarm_threshold,merge_consecutive,fp_denominator
0,_pol0,forecasting,None,None,None,None,None,0.0,None,None,...,-0.04,0.04,"{'policy': {'sph_s': 300, 'sop_s': 600, 'caden...",300,600,30,600,0.6,True,interictal
1,_pol1,forecasting,None,None,None,None,None,0.0,None,None,...,-0.04,0.04,"{'policy': {'sph_s': 300, 'sop_s': 600, 'caden...",300,600,60,600,0.6,True,interictal
2,_pol2,forecasting,None,None,None,None,None,0.0,None,None,...,-0.04,0.04,"{'policy': {'sph_s': 300, 'sop_s': 600, 'caden...",300,600,120,600,0.6,True,interictal
3,_pol3,forecasting,None,None,None,None,None,0.0,None,None,...,-0.04,0.04,"{'policy': {'sph_s': 300, 'sop_s': 600, 'caden...",300,600,300,600,0.6,True,interictal


## Where to next

- **`bridge.sample_to_alarm`** — analytic conversion bounds when only one regime was published.
- **`epileval.papers.andrade2024.metrics(...)`** — reproduces the side-by-side sample-vs-alarm panel from the Andrade 2024 paper on the same predictor.
- **`epileval.surrogates`** — registry of chance-baseline alarm generators (`poisson`, `periodic`, `circadian`, `multidien`, ...) for stricter IoC nulls.